In [1]:
import csv
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
base = "CME"
threshold = 3.5

attn = 0.4

file_CME = "Input Data/ap2_CME.csv"
file_Dino = f"Input Data/attn_{attn}/ap2_attn_{attn}.csv"

In [3]:
df_CME = pd.read_csv(file_CME, header = None)
M_CME = df_CME.to_numpy()
display(df_CME)
M_CME[:, 2] += M_CME[:, 1]-1
M_CME[:, 5] = 169 - M_CME[:, 5]

df_Dino = pd.read_csv(file_Dino)
df_Dino = df_Dino[["particle","t_starting","t","x","y","z","intensity_raw","track_length","feature_0", "feature_1","feature_2"]]
display(df_Dino)

M_Dino = df_Dino.to_numpy()
M_Dino[:, 1:3] += 1

if base == "CME":
    comp_list = np.zeros((len(M_CME),22), dtype=object)
    M_x = M_CME
    M_y = M_Dino
    sec = "Dino"
if base == "Dino":
    comp_list = np.zeros((len(M_Dino),22), dtype=object)
    M_x = M_Dino
    M_y = M_CME
    sec = "CME"

,0,1,2,3,4,5,6,7
0,2,1,1,291.30,226.51,147.760,229.960,99
1,2,1,2,291.98,226.98,148.080,242.520,99
2,2,1,3,291.70,226.58,148.100,252.340,99
3,2,1,4,291.69,227.07,148.160,272.540,99
4,2,1,5,292.56,227.12,148.270,246.290,99
...,...,...,...,...,...,...,...,...
124431,10789,98,2,528.42,544.83,14.327,119.250,2
124432,10790,98,1,563.88,119.02,13.219,32.322,2
124433,10790,98,2,563.70,119.53,14.368,34.478,2
124434,10791,98,1,523.25,139.19,15.346,29.372,2


,particle,t_starting,t,x,y,z,intensity_raw,track_length,feature_0,feature_1,feature_2
0,23,0,0,265.60117,339.69970,2.685377,122.7500,97,4.445,-6.2700,-4.950
1,23,0,1,266.07380,339.40054,4.732482,71.4400,97,7.670,-12.0700,-2.697
2,23,0,2,265.52725,339.01978,4.762420,70.5000,97,9.305,-12.3300,-3.398
3,23,0,3,266.34134,338.72610,4.893223,57.7500,97,11.500,-12.9800,-4.360
4,23,0,4,266.65646,339.07492,4.624000,65.4400,97,10.984,-12.6400,-2.676
...,...,...,...,...,...,...,...,...,...,...,...
214911,28546,93,96,487.82030,615.72960,122.027000,46.4400,4,36.220,0.7583,4.870
214912,28549,93,93,470.43076,658.63074,118.022736,12.5200,4,29.980,4.4650,5.120
214913,28549,93,94,470.20070,658.48470,118.113170,11.2800,4,30.830,3.9570,5.133
214914,28549,93,95,469.20422,659.81055,119.211050,-5.6130,4,28.250,0.5460,3.771


In [4]:
t_vec = defaultdict(list)
multi_match_list = []
for vec_y in M_y:
    t_vec[vec_y[2]].append(vec_y)

for t_val in t_vec:
    t_vec[t_val] = np.array(t_vec[t_val])

for i, vec_x in enumerate(M_x):
    t_val = vec_x[2]
    y_group = t_vec[t_val]
    diffs = y_group[:, 3:6] - vec_x[3:6]
    dists = np.linalg.norm(diffs, axis=1)
    min_idx = np.argmin(dists)
    min_dist = dists[min_idx]
    best_vec_y = y_group[min_idx]
    comp_list[i, 0:8] = vec_x
    comp_list[i, 8:19] = best_vec_y
    comp_list[i, 19] = min_dist

    below = np.where(dists < threshold)[0]
    if len(below) > 0:
        y_ids = ",".join(str(int(x)) for x in y_group[below, 0])
        dists_str = ",".join(f"{dists[j]:.2f}" for j in below)
    else:
        y_ids = ""
        dists_str = ""
    comp_list[i, 20] = y_ids
    comp_list[i, 21] = dists_str

multi_match_array = np.array(multi_match_list, dtype=object)

TypeError: list indices must be integers or slices, not tuple

In [ ]:
df_comp = pd.DataFrame(comp_list, columns = [f"ID ({base})",f"t_start ({base})", "t", f"x ({base})", f"y ({base})", 
                                             f"z ({base})", f"FI ({base})", f"Track Length ({base})", f"ID ({sec})", 
                                             f"t_start ({sec})", "t_ig", f"x ({sec})", f"y ({sec})", f"z ({sec})", 
                                             f"FI ({sec})", f"Track Length ({sec})", "Feature 0","Feature 1","Feature 2","Distance",f"Multi ID ({sec})", f"Multi Distance ({sec})"])

df_comp = df_comp.drop("t_ig", axis=1)
df_comp = df_comp[[f"ID ({base})",f"ID ({sec})", f"x ({base})", f"y ({base})", 
                   f"z ({base})", f"x ({sec})", f"y ({sec})", f"z ({sec})", "t",
                   f"t_start ({base})", f"t_start ({sec})",f"FI ({base})", 
                   f"FI ({sec})",f"Track Length ({base})", f"Track Length ({sec})", "Feature 0","Feature 1","Feature 2","Distance",f"Multi ID ({sec})", f"Multi Distance ({sec})"]] 
df_sorted = df_comp.sort_values(by=[f"ID ({base})","t"], ascending=[True, True])

In [ ]:
#df_sorted.to_csv(f"Base Output/Comparison.csv", index=False)
df_sorted.to_csv(f"DC_attn{attn}}.csv", index=False)

In [ ]:
df_sorted

In [ ]:
IDs = [491,497,499,519,524,528,541,554,556,557,560,573,574,597,598,623,661,670,674,683,685,711,716,718,722,726,739,789,805,838,845,864,874,884,889,893,932,939,948,1006,1065,1084,1122,1125,1130,1150,1200,1212,1264,1281,1284,1291,1322,1330,1336,1282]

filtered = df_sorted[df_sorted["ID (CME)"].isin(IDs)]
display(filtered)